# Math II: Linear algebra for machine learning

Data is a **matrix** (rows = examples, columns = features), a model is (mostly) a **matrix product**, and an adversarial
perturbation is a **vector** with a length. Everything here is checked with NumPy.

Contents: vectors, norms and distances (the *budget* of an attack), dot product and cosine similarity (text!), matrices as
transformations, solving linear systems and least squares (linear regression in one line), eigenvectors and PCA, SVD and
low-rank structure.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

plt.rcParams["figure.dpi"] = 100
rng = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)

## 1. Vectors, norms and distances

A message, a file or a network flow becomes a **feature vector** $\mathbf{x}\in\mathbb{R}^d$. Its *size* is measured by a **norm**:

| norm | formula | meaning |
|:--|:--|:--|
| $\ell_0$ | number of non-zero entries | *how many* features changed |
| $\ell_1$ | $\sum_i\lvert x_i\rvert$ | total change |
| $\ell_2$ | $\sqrt{\sum_i x_i^2}$ | Euclidean length |
| $\ell_\infty$ | $\max_i\lvert x_i\rvert$ | the largest single change |

**Why it matters for attacks:** an attacker's *budget* is a bound on $\lVert\boldsymbol\delta\rVert_p$ of the perturbation
$\boldsymbol\delta$. Changing 3 words is an $\ell_0$ budget; changing every pixel by at most 2/255 is an $\ell_\infty$ budget.

In [ ]:
delta = np.array([0.0, 3.0, 0.0, -4.0, 0.0, 0.5])
print("l0  =", np.count_nonzero(delta))
print("l1  =", np.linalg.norm(delta, 1))
print("l2  =", np.linalg.norm(delta, 2))
print("linf=", np.linalg.norm(delta, np.inf))

# unit balls of the three norms in 2-D
t = np.linspace(0, 2 * np.pi, 400)
circle = np.c_[np.cos(t), np.sin(t)]
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(*circle.T, label=r"$\ell_2$")
ax.plot(*(circle / np.abs(circle).sum(1, keepdims=True)).T, label=r"$\ell_1$")
ax.plot(*(circle / np.abs(circle).max(1, keepdims=True)).T, label=r"$\ell_\infty$")
ax.set(aspect="equal", title="unit balls: all perturbations of size <= 1")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## 2. Dot product and cosine similarity

$\mathbf{a}\cdot\mathbf{b}=\sum_i a_ib_i=\lVert\mathbf{a}\rVert\lVert\mathbf{b}\rVert\cos\theta$. The **cosine similarity**
$\cos\theta$ ignores length and compares only *direction*: two messages with the same word proportions are identical however long.
A **linear classifier** is a dot product: $z=\mathbf{w}\cdot\mathbf{x}+b$ (the logistic regression of the spam series, notebook 03).

In [ ]:
vocab = ["free", "call", "now", "meeting", "tomorrow", "prize"]
docs = {
    "spam A": np.array([2, 1, 1, 0, 0, 1]),
    "spam B": np.array([1, 1, 2, 0, 0, 1]),
    "ham C": np.array([0, 0, 0, 2, 1, 0]),
    "spam A x3": np.array([6, 3, 3, 0, 0, 3]),
}


def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


for other in ("spam B", "ham C", "spam A x3"):
    print(
        f"cos(spam A, {other:9}) = {cosine(docs['spam A'], docs[other]):.3f}   euclidean = {np.linalg.norm(docs['spam A'] - docs[other]):.2f}"
    )

`spam A x3` is the same message repeated three times: cosine 1.0 (same direction) but a large Euclidean distance. This is why
text is L2-normalised before comparing (TF-IDF) and why *length* is a weak signal.

## 3. Matrices are transformations

A matrix $A\in\mathbb{R}^{m\times n}$ maps $\mathbb{R}^n\to\mathbb{R}^m$ by $\mathbf{x}\mapsto A\mathbf{x}$. A neural-network layer is
$\mathbf{h}=\phi(W\mathbf{x}+\mathbf{b})$: a matrix product plus a non-linearity. Composition of transformations is **matrix
multiplication** (order matters: $AB\ne BA$).

In [ ]:
theta = np.deg2rad(30)
rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
stretch = np.diag([2.0, 0.5])
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]], dtype=float).T

fig, ax = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for a, (name, M) in zip(
    ax, (("identity", np.eye(2)), ("rotate 30°", rotation), ("stretch then rotate: R @ S", rotation @ stretch))
):
    out = M @ square
    a.plot(*square, "0.7")
    a.plot(*out, "C0")
    a.set(title=name, aspect="equal")
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print("R@S == S@R ?", np.allclose(rotation @ stretch, stretch @ rotation))

## 4. Linear systems and least squares

**Linear regression is a linear-algebra problem.** Stack $n$ examples in $X\in\mathbb{R}^{n\times d}$ (a column of ones for the
intercept) and targets in $\mathbf{y}$. The weights that minimise $\lVert X\mathbf{w}-\mathbf{y}\rVert_2^2$ satisfy the **normal
equations**

$$
X^\top X\,\mathbf{w}=X^\top\mathbf{y}\quad\Rightarrow\quad \mathbf{w}=(X^\top X)^{-1}X^\top\mathbf{y}
$$

(never invert explicitly: solve the system, or use `lstsq` / QR / SVD, which are numerically stable).

In [ ]:
n = 200
x = rng.uniform(-3, 3, n)
y = 2.5 * x - 1.0 + rng.normal(0, 1.0, n)
X = np.c_[np.ones(n), x]

w_normal = np.linalg.solve(X.T @ X, X.T @ y)
w_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]
print("normal equations :", w_normal)
print("lstsq (SVD)      :", w_lstsq)
print("condition number of X^T X:", np.linalg.cond(X.T @ X))

The next notebooks solve the *same* problem with gradient descent and PSO: three algorithms, one answer. Solving exactly is
impossible when $d$ is huge or the model is non-linear, which is why we need optimisation.

## 5. Eigenvectors, covariance and PCA

An **eigenvector** of $A$ is a direction it only *stretches*: $A\mathbf{v}=\lambda\mathbf{v}$. For the **covariance matrix**
$\Sigma=\frac1{n-1}X_c^\top X_c$ of centred data, the eigenvectors are the **principal components** (directions of maximal
variance) and the eigenvalues the variance along them. **PCA** projects on the top-$k$ components.

In [ ]:
A = rng.multivariate_normal([0, 0], [[3.0, 1.8], [1.8, 1.2]], size=500)
Xc = A - A.mean(axis=0)
cov = Xc.T @ Xc / (len(Xc) - 1)
vals, vecs = np.linalg.eigh(cov)
order = np.argsort(vals)[::-1]
vals, vecs = vals[order], vecs[:, order]
print("eigenvalues (variance along PCs):", vals, "  explained ratio:", vals / vals.sum())

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.scatter(*Xc.T, s=6, alpha=0.4)
for lam, v in zip(vals, vecs.T):
    ax.arrow(0, 0, *(2 * np.sqrt(lam) * v), color="r", width=0.04)
ax.set(aspect="equal", title="principal components")
plt.tight_layout()
plt.show()

## 6. SVD and low-rank structure

Any matrix factorises as $X=U\Sigma V^\top$. Keeping the $k$ largest singular values gives the **best rank-$k$ approximation**
(Eckart-Young). This is the idea behind PCA, latent semantic analysis (text) and *compression* of a model or an image: real data
has far fewer degrees of freedom than its dimension. We use a synthetic "term-document" matrix with 3 hidden topics plus noise.

In [ ]:
topics = rng.random((3, 60))  # 3 topics over 60 words
mix = rng.dirichlet(np.ones(3), size=300)  # 300 documents
M = mix @ topics + 0.02 * rng.normal(size=(300, 60))

U, s, Vt = np.linalg.svd(M, full_matrices=False)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.semilogy(s, "o-")
ax.set(xlabel="index", ylabel="singular value", title="3 topics: the spectrum drops after 3 values")
plt.tight_layout()
plt.show()

for k in (1, 2, 3, 10):
    Mk = (U[:, :k] * s[:k]) @ Vt[:k]
    print(
        f"rank {k:2d}: relative error {np.linalg.norm(M - Mk) / np.linalg.norm(M):.3f}   stored numbers {k * (M.shape[0] + M.shape[1] + 1)} vs {M.size}"
    )

## Exercises

1. Find a vector $\boldsymbol\delta$ with $\lVert\boldsymbol\delta\rVert_\infty\le 0.1$ that changes $\mathbf{w}\cdot\mathbf{x}$ the most, for $\mathbf{w}=(3,-2,0.5,-1)$. Compare with the best $\lVert\boldsymbol\delta\rVert_2\le 0.1$. (Hint: $\boldsymbol\delta=\varepsilon\,\mathrm{sign}(\mathbf{w})$ is the FGSM direction.)
2. Fit a quadratic $y=ax^2+bx+c$ with `lstsq` by adding a column $x^2$ to $X$. Is it still *linear* regression? Why?
3. Whiten the data of section 5 (multiply by $V\Lambda^{-1/2}$) and check that the new covariance is the identity.
4. Compute the rank-$k$ error curve for a real image or for a TF-IDF matrix of the SMS corpus.